# Preproccessing of the dataset

In [ ]:
from PIL import Image

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from keras import Input, Sequential, layers, applications

SEED = 42
CLASS_NAMES = [
    "Ascaris lumbricoides",
    "Capillaria philippinensis",
    "Enterobius vermicularis",
    "Fasciolopsis buski",
    "Hookworm egg",
    "Hymenolepis diminuta",
    "Hymenolepis nana",
    "Opisthorchis viverrine",
    "Paragonimus spp",
    "Taenia spp. egg",
    "Trichuris trichiura",
]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 224

# ---------------------------------------
# Loading the Data
# ---------------------------------------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data
!cp /content/drive/MyDrive/egg_hunt/crops_224.npz /content/data/

In [ ]:
import numpy as np

CACHE = '/content/data/crops_224.npz'

data = np.load(CACHE)
X_train, y_train, X_test, y_test = data['X_train'], data['y_train'], data['X_test'], data['y_test']
del data
print(X_train.shape, X_train.dtype, y_train.shape)

# ---------------------------------------
# Model Creation
# ---------------------------------------

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)
del X_train, y_train          # free ~1.6 GB

pd.DataFrame({"train": y_tr.sum(axis=0).astype(int),
              "val":   y_val.sum(axis=0).astype(int)}, index=CLASS_NAMES)

In [ ]:
# ------------------------------------------------------------------------------
# BUILDING THE BLOCKS
# ------------------------------------------------------------------------------

def augmentation_block(seed=None) -> Sequential:
    """Random changes during training only (inactive in predict/evaluate).
    Runs on 0–255 values."""
    return Sequential([
        layers.RandomFlip("horizontal_and_vertical", seed=seed),
        layers.RandomRotation(0.25, fill_mode="constant", seed=seed),
        layers.RandomZoom(0.1, fill_mode="constant", seed=seed),
        layers.RandomContrast(0.2, seed=seed),
        layers.RandomBrightness(0.2, value_range=(0, 255), seed=seed),
    ], name="augmentation")


def rescaling_block() -> Sequential:
    return Sequential([layers.Rescaling(1.0 / 255)], name="rescaling")


def cnn_block() -> Sequential:
     """Same conv blocks as test_baseline_cnn, so scores stay comparable."""
     return Sequential([
         layers.Conv2D(32, 3, padding="same", activation="relu"),
         layers.MaxPooling2D(2),
         layers.Conv2D(64, 3, padding="same", activation="relu"),
         layers.MaxPooling2D(2),
         layers.Conv2D(128, 3, padding="same", activation="relu"),
         layers.MaxPooling2D(2),
     ], name="cnn")

def conv_block(filters):
    return [
        layers.Conv2D(filters, 3, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(2),
    ]

def classifier_flatten_block(num_classes: int = NUM_CLASSES) -> Sequential:
    return Sequential([
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ], name="classifier")

def classifier_global_average(num_classes: int = NUM_CLASSES):
    return Sequential([
        layers.GlobalAveragePooling2D(),  # instead of Flatten -> 12.8M params gone
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation="softmax")
    ], name="classifier")

# ------------------------------------------------------------------------------
# BUILDING THE MODELS
# ------------------------------------------------------------------------------

def build_model(img_size=IMG_SIZE, num_classes=NUM_CLASSES,
                augment=False, seed=None, lr=1e-3):
    """Raw uint8 images (n, 224, 224, 3) in -> class probabilities (n, 11) out."""
    model = Sequential(name="egg_classifier_cnn")
    model.add(Input(shape=(img_size, img_size, 3)))
    if augment:
        model.add(augmentation_block(seed))
    model.add(rescaling_block())

    model.add(cnn_block())

    model.add(classifier_flatten_block(num_classes))
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy", "Recall", "Precision", "AUC"]
        )
    return model

def build_model_global_average(img_size=IMG_SIZE, num_classes=NUM_CLASSES, augment=False, seed=None):
    model = Sequential(name="egg_classifier_cnn")
    model.add(Input(shape=(img_size, img_size, 3)))
    if augment:
        model.add(augmentation_block(seed))
    model.add(rescaling_block())

    for f in (32, 64, 128, 256):                 # 4 blocks instead of 3
        for layer in conv_block(f):
            model.add(layer)

    model.add(classifier_global_average(num_classes))
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy", "Recall", "Precision", "AUC"]
        )
    return model

def build_model_efficientNet(img_size=IMG_SIZE, num_classes=NUM_CLASSES,
                         augment=False, seed=None, lr=1e-3):
    """Raw uint8 images (n, 224, 224, 3) in -> class probabilities out.
    EfficientNet does its own rescaling, so NO Rescaling(1/255) here."""
    base = applications.EfficientNetB0(
        include_top=False, weights="imagenet",
        input_shape=(img_size, img_size, 3),
    )
    base.trainable = False            # phase 1: only train the new head

    model = Sequential(name="egg_classifier")
    model.add(Input(shape=(img_size, img_size, 3)))
    if augment:
        model.add(augmentation_block(seed))
    model.add(base)
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(num_classes, activation="softmax"))
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy", "Recall", "Precision", "AUC"]
        )

    return model


In [ ]:
model = build_model(augment=False, seed=SEED)
model.summary()

#model = build_model_global_average(augment=False, seed=SEED)
#model.summary()

# model = build_model_efficientNet(augment=False, seed=SEED)
# model.summary()

In [ ]:
callbacks = [
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7)
        ]

# history = model.fit(
#      X_tr, y_tr,
#      validation_data=(X_val, y_val),
#      epochs=40,
#      batch_size=32,
#      callbacks=callbacks,
# )

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(history.history[metric], label="train")
    ax.plot(history.history[f"val_{metric}"], label="validation")
    ax.set_title(metric); ax.set_xlabel("epoch"); ax.legend()
plt.show()

In [ ]:
# Final evaluation on the test set
scores = model.evaluate(X_test, y_test, return_dict=True)
print({k: round(v, 4) for k, v in scores.items()})

y_true = y_test.argmax(axis=1)
y_pred = model.predict(X_test).argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASS_NAMES,
                                        xticks_rotation=90, normalize="true", values_format=".2f")
plt.show()